# Estrutura do banco de dados

Este notebook conecta no PostgreSQL local usando variáveis de ambiente e imprime a estrutura de schemas, tabelas e colunas.

In [4]:
from pathlib import Path

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent

import os
import pandas as pd
from sqlalchemy import inspect
from sql_utils import build_postgres_engine

engine = build_postgres_engine(
    "localhost",
    int(os.environ.get("POSTGRES_PORT", 5432)),
    os.environ["POSTGRES_DB"],
    os.environ["POSTGRES_USER"],
    os.environ["POSTGRES_PASSWORD"],
)


## Listar schemas e tabelas

Esta célula percorre os schemas disponíveis no banco, ignora schemas internos do PostgreSQL e imprime apenas os nomes das tabelas encontradas em cada schema.

In [5]:
inspector = inspect(engine)
ignored_schemas = {"information_schema", "pg_catalog", "pg_toast"}

for schema_name in sorted(inspector.get_schema_names()):
    if schema_name in ignored_schemas or schema_name.startswith("pg_"):
        continue

    table_names = sorted(inspector.get_table_names(schema=schema_name))
    if not table_names:
        continue

    print(f"schema: {schema_name}")
    for table_name in table_names:
        print(f"  table: {table_name}")
    print()


schema: raw
  table: APM_Demografico
  table: Pib_municipal
  table: SENASP
  table: age_range_ibge
  table: disk100
  table: dist_renda_ibge



## Ver colunas de uma tabela específica

Esta célula permite escolher um schema e uma tabela em `selected_schema_name` e `selected_table_name`, depois imprime as colunas dessa tabela com tipo e indicação de nulidade.

In [6]:
selected_schema_name = "raw"
selected_table_name = "SENASP"

print(f"schema: {selected_schema_name}")
print(f"table: {selected_table_name}")

for column in inspector.get_columns(selected_table_name, schema=selected_schema_name):
    nullable = "nullable" if column["nullable"] else "not null"
    print(f"  - {column['name']}: {column['type']} ({nullable})")


schema: raw
table: SENASP
  - id: VARCHAR (not null)
  - uf: TEXT (not null)
  - municipio: TEXT (not null)
  - evento: TEXT (not null)
  - data_referencia: TIMESTAMP (not null)
  - agente: TEXT (nullable)
  - arma: TEXT (nullable)
  - faixa_etaria: TEXT (nullable)
  - feminino: BIGINT (not null)
  - masculino: BIGINT (not null)
  - nao_informado: BIGINT (not null)
  - total_vitima: BIGINT (not null)
  - total: DOUBLE PRECISION (nullable)
  - total_peso: DOUBLE PRECISION (nullable)
  - abrangencia: TEXT (not null)


## Montar estrutura em DataFrame

Esta célula transforma a estrutura de schemas, tabelas e colunas em um DataFrame chamado `database_structure`, útil para filtrar, ordenar ou pesquisar metadados do banco.

In [8]:
rows = []

for schema_name in sorted(inspector.get_schema_names()):
    if schema_name in ignored_schemas or schema_name.startswith("pg_"):
        continue

    for table_name in sorted(inspector.get_table_names(schema=schema_name)):
        primary_key_columns = set(
            inspector.get_pk_constraint(table_name, schema=schema_name).get("constrained_columns", [])
        )

        for column_position, column in enumerate(
            inspector.get_columns(table_name, schema=schema_name),
            start=1,
        ):
            rows.append(
                {
                    "schema_name": schema_name,
                    "table_name": table_name,
                    "column_position": column_position,
                    "column_name": column["name"],
                    "column_type": str(column["type"]),
                    "primary_key": column["name"] in primary_key_columns,
                }
            )

database_structure = pd.DataFrame(rows)
database_structure


,schema_name,table_name,column_position,column_name,column_type,primary_key
0,raw,APM_Demografico,1,id,VARCHAR,True
1,raw,APM_Demografico,2,CD_MUN,BIGINT,False
2,raw,APM_Demografico,3,NM_MUN,TEXT,False
3,raw,APM_Demografico,4,quantidade_de_moradores,BIGINT,False
4,raw,APM_Demografico,5,sexo_masculino,BIGINT,False
...,...,...,...,...,...,...
179,raw,dist_renda_ibge,2,"Brasil, Unidade da Federação e Município",TEXT,False
180,raw,dist_renda_ibge,3,Classes de rendimento nominal mensal de todos ...,TEXT,False
181,raw,dist_renda_ibge,4,Total,TEXT,False
182,raw,dist_renda_ibge,5,Homens,TEXT,False
